# Experiment

## Import libraries

In [79]:
import pandas as pd

file_path = "fdi_rd_vietstock_2000_2025.csv"

df = pd.read_csv(file_path)

# Set indicator names as lowercase with underscores
# df["Chỉ tiêu"] = (
#     df["Chỉ tiêu"].str.lower().str.replace(",", "").str.replace(" ", "_")
# )

column_name = "Chỉ tiêu"

df[column_name] = (
    df[column_name]
    .str.lower()
    .str.replace(
        r"[^a-z0-9_\s-]", "", regex=True
    )  # remove everything except letters, numbers, underscore, space
    .str.replace(r"[\s-]+", "_", regex=True)  # replace any whitespace with underscore
)

id_vars = ["Chỉ tiêu", "Đơn vị tính"]

# Melt from wide to long format
df = df.melt(
    id_vars=id_vars,
    var_name="month_str",
    value_name="value",
)

# Clean numeric values
df["value"] = df["value"].astype(str).str.replace(",", "", regex=False)
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Extract year and month
df["date"] = pd.to_datetime(df["month_str"], errors="coerce")

# Drop rows where date couldn't be parsed
df = df.dropna(subset=["date"])

# Extract numeric year, month
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year

# Use pivot_table with first() to handle duplicates
df = df.pivot_table(
    index=["year", "month"],
    columns=id_vars[0],
    values="value",
    aggfunc="first",
).reset_index()

# Sort by year and month
df = df.sort_values(["year", "month"]).reset_index(drop=True)

# Fill missing values with 0
df.fillna(0, inplace=True)

df

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_22884\2969165376.py:37: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["month_str"], errors="coerce")


Chỉ tiêu,year,month,fdi_disbursement,register
0,2009,1,0.31,2.30
1,2009,2,0.69,3.00
2,2009,3,1.16,0.70
3,2009,4,0.04,0.40
4,2009,5,0.60,0.30
...,...,...,...,...
193,2025,3,2.01,4.08
194,2025,4,1.78,2.84
195,2025,5,2.16,4.57
196,2025,6,2.82,3.13


In [80]:
df.columns

Index(['year', 'month', 'fdi_disbursement', 'register'], dtype='object', name='Chỉ tiêu')